# Defender xPoints breakdown

Standalone notebook: top 10 defenders by expected points, split by the point-type each xPoints comes from (goals, assists, clean sheets, defensive contribution, goals-conceded penalty, appearance points).

Run jupyter from the repo root so `import fpl_v2` resolves (`uv run jupyter lab`).

In [1]:
# Ensure the repo root (the dir containing fpl_v2/) is importable, whatever the launch dir.
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / 'fpl_v2').is_dir() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [2]:
import plotly.graph_objects as go

from fpl_v2 import pipeline

[07/26/26 21:03:04] INFO     No custom team name replacements found. You can configure these in       ]8;id=8947242;file:///home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=8947243;file:///home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/soccerdata/_config.py#91\91]8;;\
                             /home/peter/soccerdata/config/teamname_replacements.json.                             

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=8947249;file:///home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=8947250;file:///home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/soccerdata/_config.py#189\189]8;;\
                             /home/peter/soccerdata/config/league_dict.json.                                       

In [3]:
forecast = pipeline.build_forecast()  # cached data; pipeline.build_forecast(refresh=True) to re-pull live

defenders = forecast[forecast['position'] == 'DEF'].sort_values('xPoints', ascending=False).head(10)
defenders[['web_name', 'team_name', 'xG', 'xAG', 'xClean', 'xBadGames', 'expected_defcon_points', 'expected_appearance_points', 'xPoints']]

                    INFO     Saving cached data to                                                   ]8;id=8947257;file:///home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=8947258;file:///home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /home/peter/coding/football_stuff/fpl_v2/data/raw/understat_match                     

[2026-07-26 21:03:04] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: /home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/tls_requests/bin/tls-client-xgo-1.13.1-linux-amd64.so


                    INFO     Successfully loaded TLS library:                                      ]8;id=8947265;file:///home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/tls_requests/models/libraries.py\libraries.py]8;;\:]8;id=8947266;file:///home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/tls_requests/models/libraries.py#397\397]8;;\
                             /home/peter/coding/football_stuff/.venv/lib/python3.12/site-packages/                 
                             tls_requests/bin/tls-client-xgo-1.13.1-linux-amd64.so                                 

,web_name,team_name,xG,xAG,xClean,xBadGames,expected_defcon_points,expected_appearance_points,xPoints
241,Virgil,Liverpool,3.77,1.44,16.0,9.0,33.988759,76.0,191.928759
0,Gabriel,Arsenal,2.94,1.75,26.0,3.0,25.740922,62.0,191.844372
373,Senesi,Spurs,1.53,4.72,12.0,10.0,45.847120,74.0,179.720453
271,Guéhi,Man City,4.05,2.37,17.0,6.0,19.341800,70.0,177.857064
371,Van Hecke,Spurs,3.31,1.57,12.0,10.0,34.828128,72.0,167.064795
1,J.Timber,Arsenal,4.71,1.53,26.0,3.0,2.743052,56.0,164.005918
180,Tarkowski,Everton,2.53,2.10,9.0,12.0,41.601834,74.0,160.450255
154,Lacroix,Crystal Palace,2.45,0.83,11.0,11.0,43.652061,69.0,159.609605
270,O'Reilly,Man City,6.12,2.67,17.0,6.0,4.371557,62.0,159.015592
52,Truffert,Bournemouth,1.35,3.07,14.0,12.0,21.454273,76.0,158.223922


## Chart

`xpoints.breakdown` (folded into `pipeline.build_forecast`) splits `xPoints` into `goal_points`, `assist_points`, `clean_points`, `conceded_points`, `defcon_points` and `appearance_points`, which sum back to `xPoints`. `conceded_points` is the only negative term (the goals-conceded penalty), so it draws to the left of zero — everything else stacks to the right.

In [4]:
# Component -> (column, legend label, categorical color).
# Colors are the first six slots of the validated categorical palette (fixed order,
# never cycled), assigned in the same order the traces stack. conceded_points is the
# only negative value, so it draws left of zero — kept last so its notch sits right
# at the bar's tail, just before the net-total label.
COMPONENTS = [
    ('goal_points', 'Goals', '#2a78d6'),
    ('assist_points', 'Assists', '#eb6834'),
    ('clean_points', 'Clean sheets', '#1baf7a'),
    ('defcon_points', 'Defensive contribution', '#eda100'),
    ('appearance_points', 'Appearance points', '#e87ba4'),
    ('conceded_points', 'Goals-conceded penalty', '#008300'),
]

SURFACE = '#fcfcfb'
GRIDLINE = '#e1e0d9'
AXIS = '#c3c2b7'
INK_PRIMARY = '#0b0b0b'
INK_SECONDARY = '#52514e'
INK_MUTED = '#898781'

# Horizontal stacked bars plot bottom-to-top in row order, so sort ascending
# to put the highest total at the top.
plot_df = defenders.sort_values('xPoints', ascending=True).reset_index(drop=True)

In [5]:
fig = go.Figure()
for col, label, color in COMPONENTS:
    fig.add_trace(go.Bar(
        y=plot_df['web_name'],
        x=plot_df[col],
        name=label,
        orientation='h',
        marker=dict(color=color, line=dict(color=SURFACE, width=1.5)),
        hovertemplate=f'%{{y}}<br>{label}: %{{x:.1f}} pts<extra></extra>',
    ))

# Direct label: net total, via a zero-width trailing bar + textposition='outside' so
# Plotly computes the padding past the bar tip itself, rather than a manual x-offset.
fig.add_trace(go.Bar(
    y=plot_df['web_name'], x=[0] * len(plot_df), orientation='h',
    marker=dict(color='rgba(0,0,0,0)'),
    text=[f'{t:.0f}' for t in plot_df['xPoints']],
    textposition='outside', textfont=dict(color=INK_SECONDARY, size=12),
    showlegend=False, hoverinfo='skip', cliponaxis=False,
))

# conceded_points is negative, so it stacks left of zero — pad the axis on both sides.
left = min(0, plot_df['conceded_points'].min()) * 1.3
right = plot_df['xPoints'].max() * 1.15

fig.update_layout(
    barmode='stack',
    bargap=0.35,
    title=dict(text='Top 10 defenders by expected points', font=dict(color=INK_PRIMARY, size=18)),
    legend=dict(font=dict(color=INK_SECONDARY)),
    plot_bgcolor=SURFACE,
    paper_bgcolor=SURFACE,
    font=dict(family='system-ui, -apple-system, "Segoe UI", sans-serif', color=INK_SECONDARY),
    xaxis=dict(title='Expected points', gridcolor=GRIDLINE, zerolinecolor=AXIS,
               tickfont=dict(color=INK_MUTED), range=[left, right]),
    yaxis=dict(tickfont=dict(color=INK_PRIMARY), automargin=True),
    margin=dict(l=10, r=150, t=60, b=40),
    height=520,
)
fig.show()

## Table view
Same data as the chart, for anyone who wants exact figures rather than reading bar lengths.

In [6]:
cols = ['web_name', 'team_name'] + [c for c, _, _ in COMPONENTS] + ['xPoints']
defenders.sort_values('xPoints', ascending=False)[cols].round(1)

,web_name,team_name,goal_points,assist_points,clean_points,defcon_points,appearance_points,conceded_points,xPoints
241,Virgil,Liverpool,22.6,4.3,64.0,34.0,76.0,-9.0,191.9
0,Gabriel,Arsenal,17.6,5.2,83.6,25.7,62.0,-2.4,191.8
373,Senesi,Spurs,9.2,14.2,46.1,45.8,74.0,-9.6,179.7
271,Guéhi,Man City,24.3,7.1,62.6,19.3,70.0,-5.5,177.9
371,Van Hecke,Spurs,19.9,4.7,45.1,34.8,72.0,-9.4,167.1
1,J.Timber,Arsenal,28.3,4.6,74.6,2.7,56.0,-2.2,164.0
180,Tarkowski,Everton,15.2,6.3,35.1,41.6,74.0,-11.7,160.5
154,Lacroix,Crystal Palace,14.7,2.5,39.7,43.7,69.0,-9.9,159.6
270,O'Reilly,Man City,36.7,8.0,52.6,4.4,62.0,-4.6,159.0
52,Truffert,Bournemouth,8.1,9.2,55.3,21.5,76.0,-11.9,158.2
